In [25]:
type lambda = 
    Num of int
  | Add of lambda * lambda

  | Var of string
  | App of lambda * lambda
  | Fun of string * lambda

let f = Fun ("x", Add (Var "x", Num 1))
let e = App (f, Num 2)

type lambda =
    Num of int
  | Add of lambda * lambda
  | Var of string
  | App of lambda * lambda
  | Fun of string * lambda


val f : lambda = Fun ("x", Add (Var "x", Num 1))


val e : lambda = App (Fun ("x", Add (Var "x", Num 1)), Num 2)


In [26]:
let rec subst (e:lambda) (x:string) (v:lambda) : lambda =
  match e with
  | Num n -> Num n
  | Add (e1, e2) -> Add (subst e1 x v, subst e2 x v)
  | Var y -> if y = x then v else Var y
  | App (e1, e2) -> App (subst e1 x v, subst e2 x v)
  | Fun (y, body) -> if y = x then Fun (y, body) else Fun (y, subst body x v)

let rec eval (e:lambda) = 
  match e with
  | Num n -> Num n
  | Add (e1,e2) -> (match (eval e1, eval e2) with
                    | (Num n1, Num n2) -> Num (n1 + n2)
                    | _ -> failwith "Dynamic Typing Error: Expected integers")

  | Fun (x, e) -> Fun (x, e)
  | App (e1, e2) -> (match eval e1 with
                    | Fun (x, body) -> 
                        eval (subst body x e2) (* Needs to avoid clash with the parameter *)
                    | _ -> failwith "Dynamic Typing Error: Expected a function")
  | Var x -> failwith ("Undefined variable: " ^ x)

let _ = eval @@ App (Fun ("x", Add (Var "x", Num 1)), Num 2)

let _ = eval @@ App (Fun ("f", App (Var "f",  Num 1)), Fun ("x", Add (Var "x",  Num 2))) 

let _ = eval @@ App (Fun ("x", Num 2), App (Fun ("x", Var "y"), Num 3)) (* Does not encounter the error *)

val subst : lambda -> string -> lambda -> lambda = <fun>


val eval : lambda -> lambda = <fun>


- : lambda = Num 3


- : lambda = Num 3


- : lambda = Num 2


In [27]:
type value = 
    VNum of int
  | VClosure of string * lambda * value env
and 'a env = (string * 'a) list

let rec eval (e:lambda) (env:value env)= 
  match e with
  | Num n -> VNum n
  | Add (e1,e2) -> (match (eval e1 env, eval e2 env) with
                    | (VNum n1, VNum n2) -> VNum (n1 + n2)
                    | _ -> failwith "Dynamic Typing Error: Expected integers")

  | Fun (x, e) -> VClosure (x, e, env)
  | App (e1, e2) -> (match eval e1 env with
                    | VClosure (x, body, closure_env) -> 
                        eval (body) ((x, eval e2 env) :: closure_env) 
                    | _ -> failwith "Dynamic Typing Error: Expected a function")
  | Var x -> (match List.assoc_opt x env with
              | Some v -> v
              | None -> failwith ("Undefined variable: " ^ x))

let _ = eval (App (Fun ("x", Add (Var "x", Num 1)), Num 2)) []

let _ = eval (App (Fun ("f", App (Var "f",  Num 1)), Fun ("x", Add (Var "x",  Num 2))) ) []

let _ = eval (App (Fun ("x", Num 2), App (Fun ("x", Var "y"), Num 3))) [] 


type value = VNum of int | VClosure of string * lambda * value env
and 'a env = (string * 'a) list


val eval : lambda -> value env -> value = <fun>


- : value = VNum 3


- : value = VNum 3


error: runtime_error

In [28]:
type typ = 
    TInt
  | TFun of typ * typ

type lambda = 
    Num of int
  | Add of lambda * lambda

  | Var of string
  | App of lambda * lambda
  | Fun of string * typ * lambda

let rec typing (e:lambda) (ctx:(string * typ) list) : typ =
  match e with
  | Num n -> TInt
  | Add (e1, e2) -> 
      let t1 = typing e1 ctx in
      let t2 = typing e2 ctx in
      if t1 = TInt && t2 = TInt then TInt       
      else failwith "Type Error: Expected integers in addition"
  | Var x -> (try List.assoc x ctx with Not_found -> failwith ("Unbound variable: " ^ x))
  | Fun (x, typ, body) -> TFun (typ, typing body ((x, typ) :: ctx))
  | App (e1, e2) ->
      let fun_type = typing e1 ctx in
      let arg_type = typing e2 ctx in
      (match fun_type with
       | TFun (param_type, return_type) ->
           if param_type = arg_type then return_type
           else failwith "Type Error: Argument type does not match parameter type"
       | _ -> failwith "Type Error: Expected a function")

let _ = typing (App (Fun ("x", TInt, Add (Var "x", Num 1)), Num 2))[]

let _ = typing (App (Fun ("f", TFun (TInt, TInt), App (Var "f",  Num 1)), Fun ("x", TInt, Add (Var "x",  Num 2))))[]

let _ = typing (App (Fun ("x", TInt, Num 2), App (Fun ("x", TInt, Var "y"), Num 3)))[] (* Does not encounter the error *) 

type typ = TInt | TFun of typ * typ


type lambda =
    Num of int
  | Add of lambda * lambda
  | Var of string
  | App of lambda * lambda
  | Fun of string * typ * lambda


val typing : lambda -> (string * typ) list -> typ = <fun>


- : typ = TInt


- : typ = TInt


error: runtime_error

In [32]:
type typ = 
    TInt
  | TFun of typ * typ
  | TVar of int * (typ option) ref

type lambda = 
    Num of int
  | Add of lambda * lambda

  | Var of string
  | App of lambda * lambda
  | Fun of string * lambda

let var_count = ref 0

let new_var () : typ =
  let v = !var_count in
  incr var_count;
  TVar (v, (ref None))

let rec unify (t1:typ) (t2:typ) : bool =
  match (t1, t2) with
  | (TInt, TInt) -> true
  | (TFun (p1, r1), TFun (p2, r2)) ->
      unify p1 p2 && unify r1 r2
  | (TVar (v1, r1), TVar (v2, r2)) when v1 = v2 -> true
  | (TVar (v, r), t) | (t, TVar (v, r)) ->
      (match !r with
       | Some t' -> unify t t'
       | None -> r := Some t; true)
  | _ -> false

let rec typing (e:lambda) (ctx:typ env) : typ =
  match e with
  | Num n -> TInt
  | Add (e1, e2) -> 
      let t1 = typing e1 ctx in
      let t2 = typing e2 ctx in
      if unify t1 TInt && unify t2 TInt then TInt       
      else failwith "Type Error: Expected integers in addition"
  | Var x -> (try List.assoc x ctx with Not_found -> failwith ("Unbound variable: " ^ x))
  | Fun (x, body) -> let ptyp = new_var () in TFun (ptyp, typing body ((x, ptyp) :: ctx))
  | App (e1, e2) ->
      let fun_type = typing e1 ctx in
      let arg_type = typing e2 ctx in
      let ftyp = TFun(new_var (), new_var ()) in
      if unify fun_type ftyp then
        match ftyp with
          | TFun (param_type, return_type) -> if unify param_type arg_type then return_type
                                            else failwith "Type Error: Argument type does not match parameter type"
          | _ -> assert false (* This case should never happen *)
      else failwith "Type Error: Expected a function"

let _ = typing (App (Fun ("f", App (Var "f",  Num 1)), Fun ("x", Add (Var "x",  Num 2))))[]

let _ = typing (Fun ("x", Var "x"))[] 

let _ = typing (Fun ("f", App (Var "f",  Num 1))) []


type typ = TInt | TFun of typ * typ | TVar of int * typ option ref


type lambda =
    Num of int
  | Add of lambda * lambda
  | Var of string
  | App of lambda * lambda
  | Fun of string * lambda


val var_count : int ref = {contents = 0}


val new_var : unit -> typ = <fun>


val unify : typ -> typ -> bool = <fun>


val typing : lambda -> typ env -> typ = <fun>


- : typ = TVar (4, {contents = Some TInt})


- : typ = TFun (TVar (6, {contents = None}), TVar (6, {contents = None}))


- : typ =
TFun
 (TVar (7,
   {contents =
     Some
      (TFun (TVar (9, {contents = Some TInt}), TVar (8, {contents = None})))}),
 TVar (8, {contents = None}))
